In [1]:
from scipy.io import loadmat
import numpy as np

mat = loadmat('Ross2018_input_data.mat')
X = mat['data'].astype(np.float32)  # shape: (N, 400)
y = mat['labels'].astype(int).flatten()

# Normalize each waveform to [-1, 1]
X /= np.max(np.abs(X), axis=1, keepdims=True)

# Reshape to (N, 400, 1)
X = X[..., np.newaxis]

# Save if you want to reuse
np.save('ross_X.npy', X)
np.save('ross_y.npy', y)


In [14]:
from tensorflow.keras.models import model_from_json
from tensorflow.keras.utils import get_custom_objects
import tensorflow.keras.backend as K
import numpy as np
import h5py

# Allow Lambda deserialization manually
# Trusting the function was written as: lambda x: K.max(x, axis=1)
get_custom_objects().update({
    '<lambda>': lambda x: K.max(x, axis=1)
})

# Load JSON model
with open('model_fm.json', 'r') as f:
    model_json = f.read()

model = model_from_json(model_json, safe_mode=False)

# Load weights
model.load_weights('model_fm_best.hdf5')

# Compile
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


TypeError: model_from_json() got an unexpected keyword argument 'safe_mode'

In [15]:
from tensorflow.keras.models import load_model
import h5py

# Load the Ross et al. polarity model
model = load_model('model_fm_best.hdf5')
model.summary()

# Predict polarity using the loaded model
y_pred_prob = model.predict(X, batch_size=128)
y_pred = (y_pred_prob >= 0.5).astype(int)


EOFError: EOF read where object expected